# ASG Airlines — Silver to Gold: Flight Star Schema

**Stage:** Silver → Gold (Medallion Architecture)
**Reads from:** `silver/flights` · **Writes to:** `gold/dim_airline`, `gold/dim_route`, `gold/dim_date`, `gold/fact_flights`

## Why Gold is flight-only

The four required business KPIs (Average Flight Duration, Route-wise Traffic, Delays/Anomalies, Airline Distribution) are all computed **at flight grain** — none of them need passenger- or booking-level data. So, by design, no passenger or booking data of any kind — masked or otherwise — is carried into Gold at all. This isn't just simpler; it's a stronger privacy guarantee than masking, since Gold is what gets connected to Power BI and potentially shared more broadly: there's nothing to leak because it was never there.

## Schema shape

A standard star schema: `fact_flights` (one row per flight) joined to three dimensions — `dim_airline`, `dim_route`, `dim_date`. Each dimension gets a clean, surrogate integer key so the fact table stays lean and joins are fast in Power BI.

## Step 1 — Storage configuration & read `silver/flights`

In [1]:
import os
import pandas as pd
from deltalake import write_deltalake, DeltaTable

STORAGE_ACCOUNT_NAME = "stasgairlines01"
CONTAINER_SILVER     = "silver"
CONTAINER_GOLD       = "gold"

STORAGE_KEY = os.environ.get("ADLS_STORAGE_KEY", "")
if not STORAGE_KEY:
    try:
        import subprocess
        res = subprocess.run(
            ["az", "storage", "account", "keys", "list",
             "--account-name", STORAGE_ACCOUNT_NAME,
             "--resource-group", "rg-asg-airlines",
             "--query", "[0].value", "-o", "tsv"],
            capture_output=True, text=True, check=True
        )
        STORAGE_KEY = res.stdout.strip()
    except Exception:
        raise RuntimeError("ADLS_STORAGE_KEY not set and could not fetch.")

so = {
    "azure_storage_account_name": STORAGE_ACCOUNT_NAME,
    "azure_storage_access_key": STORAGE_KEY,
}

df_flights = DeltaTable(f"az://{CONTAINER_SILVER}/flights", storage_options=so).to_pandas()
print(f"Loaded silver/flights: {len(df_flights):,} rows")

Loaded silver/flights: 1,003 rows


## Step 2 — Build `dim_airline`

In [1]:
dim_airline = df_flights[["airline"]].drop_duplicates().reset_index(drop=True)
dim_airline.rename(columns={"airline": "airline_name"}, inplace=True)
dim_airline.insert(0, "airline_id", range(1, len(dim_airline) + 1))
print(f"dim_airline: {len(dim_airline)} rows")
dim_airline

dim_airline: 5 rows


## Step 3 — Build `dim_route`

In [1]:
dim_route = df_flights[["source", "destination"]].drop_duplicates().reset_index(drop=True)
dim_route.insert(0, "route_id", range(1, len(dim_route) + 1))
print(f"dim_route: {len(dim_route)} rows")
dim_route.head()

dim_route: 30 rows


## Step 4 — Build `dim_date`

`date_id` is a `YYYYMMDD` integer surrogate key — the standard convention for date dimensions, since it sorts and filters naturally without needing an actual date type in every BI tool.

In [1]:
dates = pd.to_datetime(df_flights["departure_time"], format='mixed').dt.date.drop_duplicates().reset_index(drop=True)
dim_date = pd.DataFrame({"full_date": dates})
dim_date["date_id"] = pd.to_datetime(dim_date["full_date"]).dt.strftime("%Y%m%d").astype(int)
dim_date["day_of_week"] = pd.to_datetime(dim_date["full_date"]).dt.day_name()
dim_date["month"] = pd.to_datetime(dim_date["full_date"]).dt.month_name()
dim_date["is_weekend"] = pd.to_datetime(dim_date["full_date"]).dt.dayofweek.isin([5, 6])
dim_date = dim_date[["date_id", "full_date", "day_of_week", "month", "is_weekend"]]
print(f"dim_date: {len(dim_date)} rows")
dim_date.head()

dim_date: 8 rows


## Step 5 — Build `fact_flights`

Joins `silver/flights` against all three dimensions on their natural keys, then renames columns to Gold's final naming convention and assigns a clean surrogate `flight_id` (the original alphanumeric flight code is preserved as `flight_id_original`, since it's still useful for lookups/debugging).

In [1]:
fact_flights = df_flights.copy()

fact_flights = fact_flights.merge(dim_airline, left_on="airline", right_on="airline_name", how="left")
fact_flights = fact_flights.merge(dim_route, on=["source", "destination"], how="left")

fact_flights["dep_date"] = pd.to_datetime(fact_flights["departure_time"], format='mixed').dt.date
fact_flights = fact_flights.merge(
    dim_date[["date_id", "full_date"]],
    left_on="dep_date", right_on="full_date", how="left"
)

# Check for orphaned foreign keys before finalizing
missing_airlines = fact_flights["airline_id"].isna().sum()
missing_routes   = fact_flights["route_id"].isna().sum()
missing_dates    = fact_flights["date_id"].isna().sum()

fact_flights.rename(columns={
    "flight_id": "flight_id_original",
    "departure_time": "departure_datetime",
    "arrival_time": "arrival_datetime",
    "is_overnight_flight": "is_overnight",
    "duration_is_anomalous": "is_anomaly"
}, inplace=True)

fact_flights.insert(0, "flight_id", range(1, len(fact_flights) + 1))

cols_fact = [
    "flight_id", "flight_id_original", "airline_id", "route_id", "date_id",
    "departure_datetime", "arrival_datetime", "duration_minutes",
    "is_overnight", "is_anomaly"
]
fact_flights = fact_flights[cols_fact]
print(f"fact_flights: {len(fact_flights)} rows")

fact_flights: 1003 rows


## Step 6 — Write Gold Delta tables

In [1]:
write_deltalake(f"az://{CONTAINER_GOLD}/dim_airline", dim_airline, mode="overwrite", schema_mode="overwrite", storage_options=so)
write_deltalake(f"az://{CONTAINER_GOLD}/dim_route", dim_route, mode="overwrite", schema_mode="overwrite", storage_options=so)
write_deltalake(f"az://{CONTAINER_GOLD}/dim_date", dim_date, mode="overwrite", schema_mode="overwrite", storage_options=so)
write_deltalake(f"az://{CONTAINER_GOLD}/fact_flights", fact_flights, mode="overwrite", schema_mode="overwrite", storage_options=so)
print("Gold tables written: dim_airline, dim_route, dim_date, fact_flights")

Gold tables written: dim_airline, dim_route, dim_date, fact_flights


## Step 7 — Validation report

Three checks matter most here: (1) no fan-out — Silver and Gold row counts must match exactly, since joining to dimensions should never duplicate a fact row; (2) no orphaned foreign keys — every fact must successfully join to all three dimensions; (3) no passenger/booking column leakage into Gold.

In [1]:
print("=" * 80)
print("GOLD STAR SCHEMA VALIDATION REPORT")
print("=" * 80)

print("\n1. Row count / fan-out check")
print(f"   silver/flights    : {len(df_flights)} rows")
print(f"   gold/fact_flights : {len(fact_flights)} rows")
print("   -> PASS" if len(df_flights) == len(fact_flights) else "   -> FAIL (row counts differ)")

print("\n2. Orphaned foreign key check")
print(f"   airline_id NULLs: {missing_airlines}  route_id NULLs: {missing_routes}  date_id NULLs: {missing_dates}")
print("   -> PASS" if (missing_airlines == 0 and missing_routes == 0 and missing_dates == 0) else "   -> FAIL")

print("\n3. Dimension cardinalities")
print(f"   dim_airline: {len(dim_airline)}  dim_route: {len(dim_route)}  dim_date: {len(dim_date)}")

print("\n4. Gold schema dump (checking for passenger/booking leakage)")
print(f"   dim_airline : {list(dim_airline.columns)}")
print(f"   dim_route   : {list(dim_route.columns)}")
print(f"   dim_date    : {list(dim_date.columns)}")
print(f"   fact_flights: {list(fact_flights.columns)}")

print("\n5. Sample fact rows (visually joined)")
display_sample = fact_flights.head(3).merge(dim_airline, on="airline_id", how="left") \
                                       .merge(dim_route, on="route_id", how="left") \
                                       .merge(dim_date, on="date_id", how="left")
for _, row in display_sample.iterrows():
    print(f"   {row['flight_id_original']}: {row['airline_name']} | {row['source']}->{row['destination']} | "
          f"{row['duration_minutes']}m | overnight={row['is_overnight']} | anomaly={row['is_anomaly']}")

print("\nGold flight star schema generation complete.")

GOLD STAR SCHEMA VALIDATION REPORT

1. Row count / fan-out check
   silver/flights    : 1003 rows
   gold/fact_flights : 1003 rows
   -> PASS

2. Orphaned foreign key check
   airline_id NULLs: 0  route_id NULLs: 0  date_id NULLs: 0
   -> PASS

3. Dimension cardinalities
   dim_airline: 5  dim_route: 30  dim_date: 8

4. Gold schema dump (checking for passenger/booking leakage)
   dim_airline : ['airline_id', 'airline_name']
   dim_route   : ['route_id', 'source', 'destination']
   dim_date    : ['date_id', 'full_date', 'day_of_week', 'month', 'is_weekend']
   fact_flights: ['flight_id', 'flight_id_original', 'airline_id', 'route_id', 'date_id',
                  'departure_datetime', 'arrival_datetime', 'duration_minutes',
                  'is_overnight', 'is_anomaly']

5. Sample fact rows (visually joined)
   AI169: Air India | DEL->BOM | 166.0m | overnight=False | anomaly=False
   SG402: SpiceJet | BLR->HYD | 132.0m | overnight=False | anomaly=False
   6F223: IndiGo | MAA->CCU | 210